In [8]:
# ============================================================
# Task 2: End-to-End ML Pipeline for Customer Churn Prediction
# ============================================================

# 1. Install dependencies
!pip install -q scikit-learn pandas joblib huggingface_hub

# 2. Imports
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score
import joblib
from huggingface_hub import login, HfApi, create_repo
import os
from getpass import getpass

# 3. Load dataset (working URL)
url = "https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print("Dataset shape:", df.shape)

# 4. Preprocessing
df.drop('customerID', axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Identify numeric and categorical columns
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = X.columns.difference(numeric_features).tolist()

# 6. Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# 7. Create full pipelines
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 8. Hyperparameter tuning with GridSearchCV
param_grid_lr = {'classifier__C': [0.1, 1.0, 10.0]}
grid_lr = GridSearchCV(lr_pipeline, param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train, y_train)

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10],
    'classifier__min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)

# 9. Choose best model
best_model = grid_rf if grid_rf.best_score_ > grid_lr.best_score_ else grid_lr
print(f"Best model: {type(best_model.best_estimator_.named_steps['classifier']).__name__}")
print(f"Best cross-validation accuracy: {best_model.best_score_:.4f}")

# Evaluate on test set
y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc:.4f}")

# 10. Save pipeline locally
joblib.dump(best_model.best_estimator_, 'churn_pipeline.pkl')
print("Pipeline saved as 'churn_pipeline.pkl'")

# ============================================================
# 11. Upload to Hugging Face Hub (secure token input)
# ============================================================

# Try to get token from Colab secret first
hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    print("HF_TOKEN not found in Colab secrets. Please enter your token manually (it will be hidden).")
    hf_token = getpass("Enter your Hugging Face WRITE token: ")

# Login
login(token=hf_token)

REPO_ID = "narmeenbilal/churn-pipeline"  # Use your username
api = HfApi()

try:
    create_repo(repo_id=REPO_ID, exist_ok=True)
    print(f"Repository ready: {REPO_ID}")
except Exception as e:
    print(f"Note (may already exist): {e}")

api.upload_file(
    path_or_fileobj="churn_pipeline.pkl",
    path_in_repo="churn_pipeline.pkl",
    repo_id=REPO_ID,
)
print(f"✅ Pipeline successfully uploaded to: https://huggingface.co/{REPO_ID}")

Dataset shape: (7043, 21)
Best model: RandomForestClassifier
Best cross-validation accuracy: 0.8032
Test accuracy: 0.7903
Pipeline saved as 'churn_pipeline.pkl'
HF_TOKEN not found in Colab secrets. Please enter your token manually (it will be hidden).
Enter your Hugging Face WRITE token: ··········
Repository ready: narmeenbilal/churn-pipeline


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  churn_pipeline.pkl          : 100%|##########| 6.15MB / 6.15MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Pipeline successfully uploaded to: https://huggingface.co/narmeenbilal/churn-pipeline
